In [3]:
import pandas as pd

In [12]:
import os
import glob

# Lấy đường dẫn thư mục hiện tại
data_folder = os.getcwd()

# Tìm tất cả các file CSV
csv_files = glob.glob(os.path.join(data_folder, '*.csv'))

# Load tất cả các dataset với tên biến trùng với tên file
for csv_file in csv_files:
    filename = os.path.basename(csv_file)
    variable_name = filename.replace('.csv', '')
    
    try:
        globals()[variable_name] = pd.read_csv(csv_file, low_memory=False)
        print(f"✓ Loaded: {variable_name} ({globals()[variable_name].shape[0]} rows, {globals()[variable_name].shape[1]} columns)")
    except Exception as e:
        print(f"❌ Error loading {variable_name}: {e}")

print(f"\nTổng cộng: {len(csv_files)} datasets.")

✓ Loaded: customers (121930 rows, 7 columns)
✓ Loaded: geography (39948 rows, 4 columns)
✓ Loaded: inventory (60247 rows, 17 columns)
✓ Loaded: orders (646945 rows, 8 columns)
✓ Loaded: order_items (714669 rows, 7 columns)
✓ Loaded: payments (646945 rows, 4 columns)
✓ Loaded: products (2412 rows, 8 columns)
✓ Loaded: promotions (50 rows, 10 columns)
✓ Loaded: returns (39939 rows, 7 columns)
✓ Loaded: reviews (113551 rows, 7 columns)
✓ Loaded: sales (3833 rows, 3 columns)
✓ Loaded: sample_submission (548 rows, 3 columns)
✓ Loaded: shipments (566067 rows, 4 columns)
✓ Loaded: web_traffic (3652 rows, 7 columns)

Tổng cộng: 14 datasets.


In [5]:
# ==================== QUESTION 1: Inter-order Gap ====================
# Ép kiểu cột order_date sang định dạng thời gian (bỏ qua các giá trị lỗi nếu có)
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

# Lọc khách hàng có > 1 đơn hàng
customer_counts = orders['customer_id'].value_counts()
multi_order_customers = customer_counts[customer_counts > 1].index

repeat_customers_orders = orders[orders['customer_id'].isin(multi_order_customers)].copy()
# Sắp xếp theo khách hàng và thời gian
repeat_customers_orders = repeat_customers_orders.sort_values(by=['customer_id', 'order_date'])

# Tính khoảng cách ngày giữa các đơn hàng
repeat_customers_orders['prev_order_date'] = repeat_customers_orders.groupby('customer_id')['order_date'].shift(1)
repeat_customers_orders['gap_days'] = (repeat_customers_orders['order_date'] - repeat_customers_orders['prev_order_date']).dt.days

# Lấy trung vị
median_gap = repeat_customers_orders['gap_days'].median()
print(f"Q1 - Trung vị số ngày (Inter-order gap): {median_gap} ngày")

Q1 - Trung vị số ngày (Inter-order gap): 144.0 ngày


In [13]:
# ==================== QUESTION 2: Best Margin Segment ====================
# Tính gross margin
products_temp = products.copy()
products_temp['gross_margin'] = (products_temp['price'] - products_temp['cogs']) / products_temp['price'].replace(0, float('nan'))

# Tìm segment có margin trung bình cao nhất
q2_ans = products_temp.groupby('segment')['gross_margin'].mean().idxmax()
print(f"Q2 - Phân khúc có tỷ suất lợi nhuận cao nhất: {q2_ans}")

Q2 - Phân khúc có tỷ suất lợi nhuận cao nhất: Standard


In [7]:
# ==================== QUESTION 3: Streetwear Return Reason ====================
returns_with_products = returns.merge(products, on='product_id', how='inner')
streetwear_returns = returns_with_products[returns_with_products['category'] == 'Streetwear']

q3_ans = streetwear_returns['return_reason'].value_counts().idxmax()
print(f"Q3 - Lý do trả hàng phổ biến nhất (Streetwear): {q3_ans}")

Q3 - Lý do trả hàng phổ biến nhất (Streetwear): wrong_size


In [ ]:
# ==================== QUESTION 4: Lowest Bounce Rate Source ====================
q4_ans = web_traffic.groupby('traffic_source')['bounce_rate'].mean().idxmin()
print(f"Q4 - Nguồn có bounce rate thấp nhất: {q4_ans}")

Q4 - Nguồn có bounce rate thấp nhất: email_campaign


In [26]:
# ==================== QUESTION 5: Promo Application Ratio ====================
promo_ratio = (order_items['promo_id'].notna().sum() / len(order_items)) * 100
print(f"Q5 - Tỷ lệ áp dụng khuyến mãi: {promo_ratio:.2f}%")

Q5 - Tỷ lệ áp dụng khuyến mãi: 38.66%


In [14]:
# ==================== QUESTION 6: Age Group with Highest Avg Orders ====================
customers_orders = orders.merge(customers[['customer_id', 'age_group']], on='customer_id', how='inner')
customers_orders = customers_orders[customers_orders['age_group'].notna()]

# Số đơn hàng (mỗi dòng là 1 đơn)
orders_per_age = customers_orders.groupby('age_group')['order_id'].nunique()
# Số khách hàng duy nhất trong mỗi nhóm
users_per_age = customers_orders.groupby('age_group')['customer_id'].nunique()

avg_orders = (orders_per_age / users_per_age)
q6_ans = avg_orders.idxmax()
print(f"Q6 - Nhóm tuổi có số đơn trung bình cao nhất: {q6_ans}")

Q6 - Nhóm tuổi có số đơn trung bình cao nhất: 55+


In [ ]:
# ==================== QUESTION 7: Highest Revenue Region ====================
# Tính doanh thu từng món: (số lượng * đơn giá) - tiền giảm giá (không modify dataframe gốc)
item_revenue = (order_items['quantity'] * order_items['unit_price']) - order_items['discount_amount'].fillna(0)

# Tổng doanh thu theo từng order_id
revenue_per_order = pd.DataFrame({
    'order_id': order_items['order_id'],
    'item_revenue': item_revenue
}).groupby('order_id')['item_revenue'].sum().reset_index()

# Join với orders để lấy zip, rồi join geography để lấy region
orders_with_geography = orders[['order_id', 'zip']].merge(geography[['zip', 'region']], on='zip', how='inner')
revenue_by_region = revenue_per_order.merge(orders_with_geography, on='order_id', how='inner')

# Tìm vùng có tổng doanh thu cao nhất
q7_ans = revenue_by_region.groupby('region')['item_revenue'].sum().idxmax()
print(f"Q7 - Vùng tạo doanh thu cao nhất: {q7_ans}")

Q7 - Vùng tạo doanh thu cao nhất: East


In [ ]:
# ==================== QUESTION 8: Most Common Cancelled Payment Method ====================
cancelled_orders = orders[orders['order_status'] == 'cancelled']
q8_ans = cancelled_orders['payment_method'].value_counts().idxmax()
print(f"Q8 - Phương thức thanh toán khi bị hủy nhiều nhất: {q8_ans}")

Q8 - Phương thức thanh toán khi bị hủy nhiều nhất: credit_card


In [11]:
# ==================== QUESTION 9: Size with Highest Return Rate ====================
# Chỉ xét kích thước: S, M, L, XL
target_sizes = ['S', 'M', 'L', 'XL']

# Đếm tổng số sản phẩm được mua theo size
items_with_products = order_items.merge(products[['product_id', 'size']], on='product_id', how='inner')
items_with_products = items_with_products[items_with_products['size'].isin(target_sizes)]
bought_by_size = items_with_products.groupby('size').size()

# Đếm tổng số sản phẩm được trả lại theo size
returns_with_products_q9 = returns.merge(products[['product_id', 'size']], on='product_id', how='inner')
returns_with_products_q9 = returns_with_products_q9[returns_with_products_q9['size'].isin(target_sizes)]
returned_by_size = returns_with_products_q9.groupby('size').size()

# Tính tỷ lệ trả hàng = (số trả / số mua) cho mỗi size
return_rate = (returned_by_size / bought_by_size).dropna()
q9_ans = return_rate.idxmax()
print(f"Q9 - Kích thước có tỷ lệ trả cao nhất: {q9_ans} (tỷ lệ: {return_rate[q9_ans]:.2%})")

Q9 - Kích thước có tỷ lệ trả cao nhất: S (tỷ lệ: 5.65%)


In [ ]:
# ==================== QUESTION 10: Installment Plan with Highest Avg Value ====================
q10_ans = payments.groupby('installments')['payment_value'].mean().idxmax()
print(f"Q10 - Kế hoạch trả góp có giá trị TB cao nhất: {q10_ans} kỳ")

Q10 - Kế hoạch trả góp có giá trị TB cao nhất: 6 kỳ
